# Assignment 4 – Apache Spark Structured Streaming

Apache Spark Structured Streaming is used to process real-time data streams. In this assignment, we use PySpark to read data from a JSON file source and process it continuously.

Checkpointing is used to maintain the progress of streaming queries and provide fault tolerance in case of failures.

The `dropDuplicates()` function is used to identify and remove duplicate records based on a selected column.

The program demonstrates:
- Reading streaming data from a file source.
- Processing data using Spark Structured Streaming.
- Implementing fault tolerance using checkpointing.
- Detecting and removing duplicate records.

In [2]:
!pip install pyspark

In [3]:
import os
import json
import time
import shutil

from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Assignment4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Started Successfully!")

Spark Session Started Successfully!


In [4]:
input_path = "/content/stream_input"
checkpoint_path = "/content/checkpoint"

shutil.rmtree(input_path, ignore_errors=True)
shutil.rmtree(checkpoint_path, ignore_errors=True)

os.makedirs(input_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

print("Folders Created Successfully!")

Folders Created Successfully!


In [5]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("marks", IntegerType(), True)
])

print("Schema Defined Successfully!")

Schema Defined Successfully!


In [6]:
stream_df = spark.readStream \
    .schema(schema) \
    .json(input_path)

print("Streaming Data Source Created!")

Streaming Data Source Created!


In [7]:
unique_df = stream_df.dropDuplicates(["id"])

print("Duplicate Detection Applied!")

Duplicate Detection Applied!


In [8]:
query = unique_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="2 seconds") \
    .start()

print("Streaming Started Successfully!")

Streaming Started Successfully!


In [9]:
data1 = [
    {"id": 1, "name": "Ananya", "marks": 85},
    {"id": 2, "name": "Rahul", "marks": 90}
]

with open(input_path + "/data1.json", "w") as f:
    for row in data1:
        f.write(json.dumps(row) + "\n")

time.sleep(5)

print("First Batch Sent Successfully!")

First Batch Sent Successfully!


In [10]:
data2 = [
    {"id": 2, "name": "Rahul", "marks": 90},
    {"id": 3, "name": "Priya", "marks": 88}
]

with open(input_path + "/data2.json", "w") as f:
    for row in data2:
        f.write(json.dumps(row) + "\n")

time.sleep(5)

print("Second Batch Sent Successfully!")

Second Batch Sent Successfully!


In [11]:
query.stop()

print("Streaming Stopped Successfully!")

Streaming Stopped Successfully!


In [12]:
data = [10, 20, 30, 20, 40, 10, 50]

duplicates = []

for i in data:
    if data.count(i) > 1 and i not in duplicates:
        duplicates.append(i)

print("Original Data:", data)
print("Duplicate Elements:", duplicates)

Original Data: [10, 20, 30, 20, 40, 10, 50]
Duplicate Elements: [10, 20]


In [13]:
spark.stop()

print("Spark Session Closed Successfully!")

Spark Session Closed Successfully!


## Conclusion

In this assignment, Apache Spark Structured Streaming was successfully implemented using PySpark. Streaming data was read from a JSON file source, checkpointing was used for fault tolerance, and duplicate records were detected and removed using the `dropDuplicates()` function. This demonstrated how Spark can process continuous data while maintaining streaming progress and reliability.